## Hybrid Retriever- Combining Dense And Sparse Retriever

In [1]:
## Import necessary libraries
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.schema import Document

C:\Users\dhruv\AppData\Local\Temp\ipykernel_27032\3400232190.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
## Step 1: Sample documents
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]

## Step 2: Dense Retriever (FAISS + HuggingFace)
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(docs, embedding_model)
dense_retriever = dense_vectorstore.as_retriever()

## Step 3: Sparse Retriever (BM25)
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k = 3 ## top- k documents to retriever

## Step 4: Combine with Ensemble Retriever
hybrid_retriever = EnsembleRetriever(
    retrievers = [dense_retriever, sparse_retriever],

    ## Provide the weightage (α) for the hybrid score
    weight = [0.7, 0.3]
)

hybrid_retriever

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000258422DE510>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x00000258422DECF0>, k=3)], weights=[0.5, 0.5])

In [3]:
## Step 5: Query and get results
query = "How can I build an application using LLMs?"
results = hybrid_retriever.invoke(query)

## Step 6: Print results
for i, doc in enumerate(results):
    print(f"\n🔹 Document {i+1}:\n{doc.page_content}")


🔹 Document 1:
LangChain helps build LLM applications.

🔹 Document 2:
Langchain can be used to develop agentic ai application.

🔹 Document 3:
Langchain has many types of retrievers.

🔹 Document 4:
Pinecone is a vector database for semantic search.


### RAG Pipeline with hybrid retriever

In [4]:
## Important necessary libraries
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

import os

groq_api_key = os.getenv("GROQ_API_KEY")

In [ ]:
# Step 5: Prompt Template
prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

## step 6-llm
# llm=init_chat_model("openai:gpt-3.5-turbo",temperature=0.2)
# llm

## LLM
llm = init_chat_model(model = "openai/gpt-oss-120b", model_provider = "groq", temperature = 0.4)
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000025843517230>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000258441E4980>, model_name='openai/gpt-oss-120b', temperature=0.4, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [6]:
## Create stuff Docuemnt Chain
document_chain = create_stuff_documents_chain(llm = llm, prompt = prompt)

## Create full RAG chain
rag_chain = create_retrieval_chain(retriever = hybrid_retriever, combine_docs_chain = document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000258422DE510>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x00000258422DECF0>, k=3)], weights=[0.5, 0.5]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below.\n\nContext:\n{context}\n\nQuestion: {input}\n')
            | ChatGroq(output_ve

In [7]:
## Ask a question
query = {"input": "How can I build an app using LLMs?"}
response = rag_chain.invoke(query)

## Get the output
print("✅ Answer:\n", response["answer"])

print("\n📄 Source Documents:")
for i, doc in enumerate(response["context"]):
    print(f"\nDoc {i+1}: {doc.page_content}")

✅ Answer:
 Below is a practical, step‑by‑step roadmap for turning a large language model (LLM) into a working application.  
I’ll lean on the tools you mentioned—**LangChain**, **agents**, **retrievers**, and **Pinecone**—but the pattern works with any LLM provider (OpenAI, Anthropic, Cohere, Llama 2, etc.).

---

## 1️⃣ Clarify the product vision  

| Question | Why it matters |
|----------|----------------|
| **What problem am I solving?** | Determines the prompt style, required knowledge sources, and UI. |
| **Is the app conversational, generative, or retrieval‑augmented?** | Guides whether you need a plain LLM chain, an agent, or a retriever‑augmented generation (RAG) pipeline. |
| **What latency / cost constraints do I have?** | Influences model size, number of API calls, and whether you cache results. |
| **Do I need to store user data or private documents?** | If yes, you’ll need a vector store (e.g., Pinecone) and possibly encryption. |

Write a one‑sentence “product statement”